# Prototyping Core + Extension Catalogs

* Experimental Registry Repo: https://github.com/dougbrn/hats-registry
* LSDB Registry Dev Branch: https://github.com/astronomy-commons/lsdb/tree/registry

## Starting with the LSDB API

In [1]:
import lsdb

In [2]:
gaia = lsdb.open_catalog('https://data.lsdb.io/hats/gaia_dr3', columns=["ra","dec"])

#append some metadata that would be innate in the proper version of this
gaia.hc_structure.catalog_info.hats_registry_id = "gaia_dr3"

gaia

,ra,dec
npartitions=2016,,
"Order: 2, Pixel: 0",double[pyarrow],double[pyarrow]
...,...,...
"Order: 3, Pixel: 766",...,...
"Order: 3, Pixel: 767",...,...


In [3]:
ext = gaia.show_extensions()
ext

extends,gaia_dr3
modality,tabular
path,https://data.lsdb.io/hats/gaia_edr3_distances
coverage,None
mirror,primary


In [4]:
gaia.load_extension(ext[0]) # or could load using "bailer-jones" directly

,ra,dec,source_id,r_med_geo,r_lo_geo,r_hi_geo,r_med_photogeo,r_lo_photogeo,r_hi_photogeo,flag,Norder,Dir,Npix
npartitions=3243,,,,,,,,,,,,,
"Order: 2, Pixel: 0",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int8[pyarrow],int64[pyarrow],int64[pyarrow]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 766",...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 767",...,...,...,...,...,...,...,...,...,...,...,...,...


## Digging into the hats-registry

In [5]:
from hats_registry import HatsRegistry

In [6]:
# Load the central registry -- from https://github.com/dougbrn/hats-registry/tree/main/registry
registry = HatsRegistry.load()

In [7]:
# Finding the registry entry for Gaia, a core catalog
registry.resolve("gaia_dr3")

CoreCatalogEntry(catalog_id='gaia_dr3', paths={'primary': 'https://data.lsdb.io/hats/gaia_dr3'}, catalog_type='core')

In [8]:
# Finding the registry entry for Bailer-Jones, an extension catalog of Gaia
gaia_ext = registry.get_extensions("gaia_dr3")
gaia_ext

# or registry.resolve("bailer-jones)

[ExtensionCatalogEntry(catalog_id='bailer-jones', paths={'primary': 'https://data.lsdb.io/hats/gaia_edr3_distances'}, catalog_type='extension', extends='gaia_dr3', modality='tabular', coverage='partial')]

In [9]:
gaia_ext[0]

ExtensionCatalogEntry(catalog_id='bailer-jones', paths={'primary': 'https://data.lsdb.io/hats/gaia_edr3_distances'}, catalog_type='extension', extends='gaia_dr3', modality='tabular', coverage='partial')

In [10]:
registry._extensions_by_core

{'gaia_dr3': [ExtensionCatalogEntry(catalog_id='bailer-jones', paths={'primary': 'https://data.lsdb.io/hats/gaia_edr3_distances'}, catalog_type='extension', extends='gaia_dr3', modality='tabular', coverage='partial')]}

## More Details with a Toy Registry

In [11]:
from lsdb import registry as lsdb_registry
from pathlib import Path

In [12]:
test_registry = HatsRegistry.from_directory(
    Path("/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/registry")
)


test_registry

In [13]:
core_entry = test_registry.resolve("test_core")
core_entry

CoreCatalogEntry(catalog_id='test_core', paths={'primary': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/primary/test_core', 'sdf': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/sdf/test_core'}, catalog_type='core')

In [14]:
core_entry.paths

{'primary': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/primary/test_core',
 'sdf': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/sdf/test_core'}

In [15]:
test_registry.get_extensions("test_core")

[ExtensionCatalogEntry(catalog_id='test_core_derived', paths={'primary': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/primary/test_core_derived', 'sdf': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/sdf/test_core_derived'}, catalog_type='extension', extends='test_core', modality='tabular'),
 ExtensionCatalogEntry(catalog_id='test_core_subset', paths={'primary': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/primary/test_core_subset', 'sdf': '/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/sdf/test_core_subset'}, catalog_type='extension', extends='test_core', modality='tabular', description='A simple extension adding the `a` column (ra, dec, a)')]

In [16]:
# some hacky patching to use this registry with LSDB
from hats_registry import get_default_ref
lsdb_registry._registry_cache[get_default_ref()] = test_registry

In [17]:
core_cat = lsdb.open_catalog(core_entry.paths["primary"])
core_cat

,ra,dec,id,b,nested
npartitions=12,,,,,
"Order: 0, Pixel: 0",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],"nested<t: [double], flux: [double], flux_error..."
...,...,...,...,...,...
"Order: 0, Pixel: 10",...,...,...,...,...
"Order: 0, Pixel: 11",...,...,...,...,...


In [18]:
ext = core_cat.show_extensions()
ext

extends,test_core
modality,tabular
path,/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/primary/test_core_derived
coverage,None
mirror,primary
extends,test_core
modality,tabular
path,/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/primary/test_core_subset
coverage,None
mirror,primary
description,"A simple extension adding the `a` column (ra, dec, a)"


In [20]:
core_cat.load_extension(ext[0]).load_extension(ext[1])

/Users/dbranton/lincc/lsdb/src/lsdb/operations/functions/crossmatch_catalog_data.py:357: RuntimeWarning: Right catalog does not have a margin cache. Results may be incomplete and/or inaccurate.
  warnings.warn(
/Users/dbranton/lincc/lsdb/src/lsdb/operations/functions/crossmatch_catalog_data.py:357: RuntimeWarning: Right catalog does not have a margin cache. Results may be incomplete and/or inaccurate.
  warnings.warn(


,ra,dec,id,b,nested,derived_value,a
npartitions=12,,,,,,,
"Order: 0, Pixel: 0",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],"nested<t: [double], flux: [double], flux_error...",double[pyarrow],double[pyarrow]
...,...,...,...,...,...,...,...
"Order: 0, Pixel: 10",...,...,...,...,...,...,...
"Order: 0, Pixel: 11",...,...,...,...,...,...,...


### Mirrors

In [21]:
core_cat_on_usdf = lsdb.open_catalog(core_entry.paths["sdf"])
core_cat_on_usdf

,ra,dec,id,b,nested
npartitions=12,,,,,
"Order: 0, Pixel: 0",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],"nested<t: [double], flux: [double], flux_error..."
...,...,...,...,...,...
"Order: 0, Pixel: 10",...,...,...,...,...
"Order: 0, Pixel: 11",...,...,...,...,...


In [22]:
ext_on_usdf = core_cat_on_usdf.show_extensions()
ext_on_usdf

extends,test_core
modality,tabular
path,/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/sdf/test_core_derived
coverage,None
mirror,sdf
extends,test_core
modality,tabular
path,/Users/dbranton/lincc/hats-registry/tests/data/fixture_registry/catalogs/sdf/test_core_subset
coverage,None
mirror,sdf
description,"A simple extension adding the `a` column (ra, dec, a)"


In [23]:
core_cat_on_usdf.load_extension(ext_on_usdf[0])

/Users/dbranton/lincc/lsdb/src/lsdb/operations/functions/crossmatch_catalog_data.py:357: RuntimeWarning: Right catalog does not have a margin cache. Results may be incomplete and/or inaccurate.
  warnings.warn(


,ra,dec,id,b,nested,derived_value
npartitions=12,,,,,,
"Order: 0, Pixel: 0",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],"nested<t: [double], flux: [double], flux_error...",double[pyarrow]
...,...,...,...,...,...,...
"Order: 0, Pixel: 10",...,...,...,...,...,...
"Order: 0, Pixel: 11",...,...,...,...,...,...
